# WP2/T2.3 Mapping units of measurement 
[Owner: C]

For every numeric attribute, map the unit of measurement to an ontological concept. 
The recommended ontology is the SI Digital Framework

*Author*: *Ambrogi Federico*

## Mapping units of measurement

This section describes the semantic annotation of the precipitation dataset by mapping all numeric variables to standardized units using controlled ontologies, and by aligning each variable with a formal unit definition. 

The primary reference ontology is the SI Digital Framework, which provides standardized representations of SI units. 

Each numeric variable physical quantity and associated unit are then mapped to corresponding ontology URIs, with preference given to SI-compliant representations. 


Where necessary, units are normalized into SI-coherent forms:
- mg/L and µg/L 
- electrical conductivity expressed in siemens per metre
- pH (dimensionless) explicitly marked

The resulting mappings are structured into a metadata payload that preserves both the original unit and its ontological representation. 



**Source**

http://si-digital-framework.org/



**Example** ![alt text](mm.png "Title")

### Workflow
To map every numerical attribute to a specific physical unit, 
we need to modify the dataset and introduce a dedicated table "units" with the following entries:
- "code": unique identifier for the unit
- "symbol": symbolic representation in IS
- "meaning": semnatic explanation of the symbol/unit
- "canonical_si": unit in canonical IS 
- "conversion_to_si": convertion numerical factor to IS,
- "ontology": url from the 

In [5]:
import os
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient
import pandas as pd
import numpy as np

from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)

load_dotenv("../.env")

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Current user:", client.whoami())
print("Database ID:", DATABASE_ID)

rashidulaminsaad
Current user: rashidulaminsaad
Database ID: bfa4385b-54a9-4ae3-b4f4-cb503d7bb016


## Creating Metadata Ontology Entries for units

To map every numerical attribute to a specific physical unit, 
we need to modify the dataset and introduce a dedicated table "units" with the following entries:
- *"code"*: unique identifier for the unit
- *"symbol"*: symbolic representation in IS
- *"meaning"*: semantic explanation of the symbol/unit
- *"canonical_si"*: unit in canonical IS 
- *"conversion_to_si"*: convertion numerical factor to IS,
- *"ontology"*: url from http://si-digital-framework.org/



In [6]:
# Creating UNITS Table

units_data = [
    {
        "unit": 1,
        "symbol": "mm",
        "meaning": "millimetre",
        "canonical_si": "metre",
        "conversion_to_si": 1e-3,
        "ontology": "http://si-digital-framework.org/SI/units/millimetre"
    },
    {
        "unit": 2,
        "symbol": "µS/cm",
        "meaning": "microsiemens per centimetre",
        "canonical_si": "siemens per metre",
        "conversion_to_si": 1e-4,
        "ontology": "http://si-digital-framework.org/SI/units/siemens-per-metre"
    },
    {
        "unit": 3,
        "symbol": "mg/L",
        "meaning": "milligrams per litre",
        "canonical_si": "kilogram per cubic metre",
        "conversion_to_si": 1e-3,
        "ontology": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre"
    },
    {
        "unit": 4,
        "symbol": "µg/L",
        "meaning": "micrograms per litre",
        "canonical_si": "kilogram per cubic metre",
        "conversion_to_si": 1e-6,
        "ontology": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre"
    },
    {
        "unit": 5,
        "symbol": "dimensionless",
        "meaning": "dimensionless quantity",
        "canonical_si": "unity",
        "conversion_to_si": 1,
        "ontology": "http://si-digital-framework.org/SI/units/unity"
    }
]

# Create dataframe
units_table_df = pd.DataFrame(units_data)

# Permanently changes the pandas settings
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

units_table = units_table_df.copy()
units_table = units_table.set_index("unit")
units_table.index.name = "unit"

for c in units_table.columns:
    units_table[c] = units_table[c].astype(str)

units_table['conversion_to_si'] = units_table['conversion_to_si'].astype(float)

print(units_table)

             symbol                      meaning              canonical_si  \
unit                                                                         
1                mm                   millimetre                     metre   
2             µS/cm  microsiemens per centimetre         siemens per metre   
3              mg/L         milligrams per litre  kilogram per cubic metre   
4              µg/L         micrograms per litre  kilogram per cubic metre   
5     dimensionless       dimensionless quantity                     unity   

      conversion_to_si  \
unit                     
1             0.001000   
2             0.000100   
3             0.001000   
4             0.000001   
5             1.000000   

                                                               ontology  
unit                                                                     
1                   http://si-digital-framework.org/SI/units/millimetre  
2            http://si-digital-framework.org/SI/

In [7]:
units_description = " Unit table using ontology mappings follows the SI Digital Framework http://si-digital-framework.org/SI/"

In [9]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    print(t.name)
    print(t.description)
    print("-----")

stations
Stations for Austrian precipitation data derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
measurement_variables
Measurement variable metadata derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
precipitation_measurements
Precipitation chemistry measurements derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----


## Table creation and Upload

In [10]:
units_results = client.create_table(
    database_id=DATABASE_ID,
    name="units",
    is_public=True,
    is_schema_public=True,
    dataframe=units_table,
    description=units_description
)

2026-05-27 10:29:21,213 root         WARNING default to 'text' for column symbol and type <class 'numpy.dtype'>
2026-05-27 10:29:21,214 root         WARNING default to 'text' for column meaning and type <class 'numpy.dtype'>
2026-05-27 10:29:21,215 root         WARNING default to 'text' for column canonical_si and type <class 'numpy.dtype'>
2026-05-27 10:29:21,216 root         WARNING default to 'text' for column ontology and type <class 'numpy.dtype'>


In [11]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    print(t.name)
    print(t.description)
    print("-----")

units
 Unit table using ontology mappings follows the SI Digital Framework http://si-digital-framework.org/SI/
-----
stations
Stations for Austrian precipitation data derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
measurement_variables
Measurement variable metadata derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----
precipitation_measurements
Precipitation chemistry measurements derived from Redl, Steinkogler and Kasper-Giebl 2021, DOI: https://doi.org/10.48436/b0g4h-rv840, license: CC BY-NC-SA 4.0.
-----


## Update SQL SCHEMA creation + Relational Diagram

**-> Updated in notebook "T2-1 _dbrepo_schema_creation"**

```
CREATE TABLE units (
    unit INTEGER PRIMARY KEY,
    symbol VARCHAR(20) NOT NULL UNIQUE,
    meaning VARCHAR(255) NOT NULL,
    canonical_si VARCHAR(50) NOT NULL
    conversion_to_si DOUBLE;
);
```

## Updating measurement_variable Table

**-> Updated in notebook "T2-1 _dbrepo_schema_creation"**

```
measurement_variable_updated = pd.DataFrame([
    {"variable_code": "NS", "label": "Precipitation amount", "unit": "1"},
    {"variable_code": "LF", "label": "Conductivity", "unit": "2"},
    {"variable_code": "pH", "label": "pH value", "unit": "5"},
    {"variable_code": "NH4", "label": "Ammonium concentration", "unit": "3"},
    {"variable_code": "Na", "label": "Sodium concentration", "unit": "3"},
    {"variable_code": "K", "label": "Potassium concentration", "unit": "3"},
    {"variable_code": "Ca", "label": "Calcium concentration", "unit": "3"},
    {"variable_code": "Mg", "label": "Magnesium concentration", "unit": "3"},
    {"variable_code": "Cl", "label": "Chloride concentration", "unit": "3"},
    {"variable_code": "NO3", "label": "Nitrate concentration", "unit": "3"},
    {"variable_code": "SO4", "label": "Sulfate concentration", "unit": "3"},
    {"variable_code": "Pb", "label": "Lead concentration", "unit": "4"},
    {"variable_code": "Cd", "label": "Cadmium concentration", "unit": "4"},
])

measurement_variables.insert(0, "variable_id", range(1, len(measurement_variables) + 1))


units_description = """
Unit table for precipitation dataset
measurement variables.

Each variable is associated with:
- original unit of measurement and its symbol
- semantic representation
- convertion to canonical SI units for derived units 
- SI-based ontology url

Ontology mappings follows the
SI Digital Framework
http://si-digital-framework.org/SI/
"""
```